In [6]:
from sklearn.metrics.pairwise import euclidean_distances
from pathlib import Path
import pandas as pd

In [57]:
path_work_dir = Path("/home/b05b01002/HDD/project_scRNAed")
path_output = path_work_dir / "outputs/notebooks/Show-Proteome-Validated-on-Seurat-UMAP/"
path_embeddings = path_work_dir / "outputs/notebooks/Show-Proteome-Validated-on-Seurat-UMAP/integrated.cca.csv"
path_cell_ident = path_work_dir / "references/ptr_tenx_batch1_rs17_curated.csv"

### Get cell identity by nearest reference

Read reference cell identity

In [34]:
reference_cell_ident = pd.read_csv(path_cell_ident, index_col=0)
reference_cell_ident.head()

,Cluster,Color
Barcode,,
AAACCTGAGCACCGCT-1,8,#E377C2
AAACCTGCAAGAGTCG-1,7,#D62728
AAACCTGCACATTCGA-1,3,#FF7F0F
AAACCTGGTAAATGAC-1,3,#FF7F0F
AAACCTGGTCATCGGC-1,1,#1F77B4


Read embeddings of all bioreps

In [11]:
embeddings = pd.read_csv(
    path_embeddings,
    index_col=0
)
embeddings.head()

,integratedcca_1,integratedcca_2,integratedcca_3,integratedcca_4,integratedcca_5,integratedcca_6,integratedcca_7,integratedcca_8,integratedcca_9,integratedcca_10,...,integratedcca_21,integratedcca_22,integratedcca_23,integratedcca_24,integratedcca_25,integratedcca_26,integratedcca_27,integratedcca_28,integratedcca_29,integratedcca_30
AAACCTGAGCACCGCT-1_1,8.553631,-6.008619,-10.468839,-5.856133,-48.376924,45.620813,2.572424,19.380106,6.624714,-20.053965,...,0.405844,0.531237,-8.313060,0.689239,3.406736,-3.707216,-0.181881,0.115383,1.401048,1.605206
AAACCTGCAAGAGTCG-1_1,-117.451413,-9.682598,-3.159338,19.702042,2.412085,3.317594,12.266135,2.274045,0.714785,2.188559,...,-0.733071,-0.135228,6.041794,4.580537,12.267207,-3.583874,-4.448090,-1.436238,0.661395,1.226619
AAACCTGCACATTCGA-1_1,15.869051,-39.146500,-2.289393,-1.813494,6.553414,8.280496,0.225023,-24.714781,3.586793,-2.418766,...,1.516785,5.197331,-3.914603,8.858530,0.322296,0.372711,4.294704,4.169678,2.518136,3.936872
AAACCTGGTAAATGAC-1_1,10.859940,-20.811009,-26.278891,0.547737,3.571729,5.792480,-2.214079,-10.171792,0.004572,1.603109,...,0.216264,1.029186,4.033681,-12.215694,1.839143,-2.549375,-1.396900,1.570853,-1.299284,-3.293464
AAACCTGGTCATCGGC-1_1,-0.468537,17.389900,7.851869,0.961913,0.063824,-3.162259,1.174197,-2.405853,-0.634247,-0.679696,...,0.973016,4.740723,-1.487619,-1.427436,0.847993,-3.265063,-0.791963,-3.914409,-3.453628,-3.225413


Calculate pairwise distances

In [50]:
distance = euclidean_distances(embeddings, embeddings.loc[["%s_1" % (i) for i in  reference_cell_ident.index]])

In [51]:
distance = pd.DataFrame(
    distance,
    index=embeddings.index,
    columns=reference_cell_ident.index
)

In [52]:
nearest_reference = distance.idxmin(axis=1)

In [59]:
cell_identity = pd.DataFrame(
    dict(
        Barcode=nearest_reference.index,
        Cluster=reference_cell_ident.loc[nearest_reference, "Cluster"].values,
    )
)
cell_identity.to_csv(
    path_output / "cell_identity_all.csv",
    index=False
)

### Plot RNA editing level on UMAP

In [67]:
from scipy.io import mmread

sample_names = [
    "ptr_tenx_batch1",
    "ptr_tenx_batch2",
    "ptr_tenx_tsv1",
    "ptr_tenx_tsv2",
    "ptr_tenx_tsv3",
    "ptr_tenx_tsv4",
    "ptr_tenx_tsv5"
]
path_barcode = path_work_dir / "outputs/Remapping/renamer/{sample}/barcodes.tsv"
path_snv_loci = path_work_dir / "outputs/VariantCalling/vawk/{sample}.snv.loci.txt"
path_alt_mtx = path_work_dir / "outputs/VariantCalling/vartrix/{sample}/alt.mtx"
path_ref_mtx = path_work_dir / "outputs/VariantCalling/vartrix/{sample}/ref.mtx"
# proteom validation results
path_proteome_validation = path_work_dir / "outputs/notebooks/Check-proteome-alignment-results/variant_annotation_6frames_peptide_validation.tsv"

In [227]:
proteome_validation = pd.read_csv(path_proteome_validation, sep = "\t")
not_synonymous = proteome_validation["CONSEQUENCE"] != "synonymous"
is_covered = proteome_validation["PEPTIDE_COVERED"] == True
validated_gene = proteome_validation.loc[not_synonymous & is_covered, "GENEID"].apply(
    lambda x: "%s.%s.v4.1" % (x.split(".")[0], x.split(".")[1])
).tolist()
validated_loci = proteome_validation.loc[not_synonymous & is_covered, ["CHROM", "POS"]].apply(
  lambda x: ":".join([str(i) for i in x]), 1
).tolist()

In [232]:
loci_to_gene = {l:"_".join([l.replace(":","-"),g]) for l, g in zip(validated_loci, validated_gene)}
loci_to_gene

{'Chr01:2167186': 'Chr01-2167186_Potri.001G028600.v4.1',
 'Chr01:29185246': 'Chr01-29185246_Potri.001G278500.v4.1',
 'Chr01:36330671': 'Chr01-36330671_Potri.001G350801.v4.1',
 'Chr01:31982741': 'Chr01-31982741_Potri.001G309500.v4.1',
 'Chr02:3803612': 'Chr02-3803612_Potri.002G055700.v4.1',
 'Chr02:2266172': 'Chr02-2266172_Potri.002G034100.v4.1',
 'Chr02:7658054': 'Chr02-7658054_Potri.002G104400.v4.1',
 'Chr03:15396250': 'Chr03-15396250_Potri.003G136400.v4.1',
 'Chr03:16693059': 'Chr03-16693059_Potri.003G155800.v4.1',
 'Chr04:6256161': 'Chr04-6256161_Potri.004G075200.v4.1',
 'Chr05:4021766': 'Chr05-4021766_Potri.005G062800.v4.1',
 'Chr05:20501622': 'Chr05-20501622_Potri.005G198900.v4.1',
 'Chr05:24310323': 'Chr05-24310323_Potri.005G249200.v4.1',
 'Chr05:24310328': 'Chr05-24310328_Potri.005G249200.v4.1',
 'Chr05:3507120': 'Chr05-3507120_Potri.005G055900.v4.1',
 'Chr06:23589873': 'Chr06-23589873_Potri.006G233200.v4.1',
 'Chr07:7008602': 'Chr07-7008602_Potri.007G061861.v4.1',
 'Chr07:93203

In [226]:
alt_frac_all = None
for idx, i in enumerate(sample_names):
    loci = pd.read_csv(path_snv_loci.__str__().replace("{sample}", i), header=None)
    is_loci_of_interest = loci[0].isin(validated_loci)
    print("Found", sum(is_loci_of_interest), "loci in sample:", i)
    barcode = pd.read_csv(path_barcode.__str__().replace("{sample}", i), header=None)
    alt_mtx_temp = mmread(path_alt_mtx.__str__().replace("{sample}", i)).tocsr()
    ref_mtx_temp = mmread(path_ref_mtx.__str__().replace("{sample}", i)).tocsr()
    alt_mtx_temp = alt_mtx_temp[is_loci_of_interest, ]
    ref_mtx_temp = ref_mtx_temp[is_loci_of_interest, ]
    cov_mtx_temp = alt_mtx_temp + ref_mtx_temp
    alt_frac_temp = alt_mtx_temp / cov_mtx_temp
    alt_frac = pd.DataFrame(alt_frac_temp.T)
    alt_frac.columns = loci.loc[is_loci_of_interest, 0]
    alt_frac.index = [f"{i}_{idx+1}" for i in barcode[0]]
    alt_frac[pd.DataFrame.sparse.from_spmatrix(cov_mtx_temp) < 5] = 0
    alt_frac[alt_frac.isna()] = 0
    if alt_frac_all is None:
        alt_frac_all = alt_frac
    else:
        alt_frac_all = pd.concat([alt_frac_all, alt_frac], sort=False)

alt_frac_all[alt_frac_all.isna()] = 0

Found 45 loci in sample: ptr_tenx_batch1
Found 17 loci in sample: ptr_tenx_batch2
Found 14 loci in sample: ptr_tenx_tsv1
Found 15 loci in sample: ptr_tenx_tsv2
Found 16 loci in sample: ptr_tenx_tsv3
Found 17 loci in sample: ptr_tenx_tsv4
Found 15 loci in sample: ptr_tenx_tsv5


In [234]:
alt_frac_all = alt_frac_all.rename(
    loci_to_gene, axis = 1
)

In [235]:
alt_frac_all.to_csv(
    path_output / "editing_level.csv"
)